In [1]:
#comparing synaptic efficacy in AN network model with different SD for number of synapses per neuron  FigS10

import traceback


import math
from scipy.signal import find_peaks
from scipy.signal import periodogram
from scipy import signal
import scipy


import shutil

import numpy as np
import pandas as pd

import gc


from scipy.integrate import odeint

import scipy.stats as st
import itertools

from datetime import datetime

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

from statannot import add_stat_annotation
import os
import sys

sys.path.append('../')
sys.path.append('../anmodel')

import anmodel

import pickle

sns.set()
sns.set_style("ticks") 
np.set_printoptions(threshold=1000)

fs = 20
fs_l = 15

In [3]:
class eval_wave_lr:
    def __init__(self, NE, NI, T, Tp, dt, T_w, v_off, drc_neu, drc_syn, tbs):
        self.N = NE + NI
        self.NE = NE
        self.NI = NI
     
        self.tbs = int(T/Tp)# tbs #100 # int(T/Tp)
        self.T = T
        self.Tp=Tp
        self.dt = dt
        self.drc_neu = drc_neu
        self.drc_syn = drc_syn
#         self.th_p = pars_lr["th_p"]
#         self.th_d = pars_lr["th_d"]
        self.df_np = np.zeros((int((T-v_off)/T_w),11))
        self.x= np.arange(0,tbs*Tp,dt)/1000
    
    def v_of_exp(self, n, ex):
        ex_in = ex+self.ex_diff
        for tb in range(self.tbs):
            v_file = self.drc_neu+ "ex"+str(ex)+"_"+"ex_in"+str(ex_in)+"_"+"N"+str(n)+"_"+str(tb)+".bin" 
            f2= open(v_file, "rb")
            rectype = np.dtype(np.float64)
            v_tb = np.fromfile(f2, dtype=rectype)
            if tb==0:
                v=v_tb
            else:
                v=np.append(v, v_tb)
            f2.close()
        return v
    
    def rho_of_exp(self, pre_syn, n):
        
        for tb in range(self.tbs):
            rho_file = self.drc_syn+"rho"+str(pre_syn)+"-"+str(n)+"_"+str(tb)+".bin"
            frho= open(rho_file, "rb")
            rectype = np.dtype(np.float64)
            rho_tb = np.fromfile(frho, dtype=rectype)
            if tb==0:
                rho=rho_tb
            else:
                rho=np.append(rho, rho_tb)
            frho.close()
        return rho
    
    def ave_rho(self, N,  start):
        try:
            v_file = self.drc_neu +"N"+str(0)+"_"+str(0)+".bin" 
            f2= open(v_file, "rb")
            rectype = np.dtype(np.float64)
            v_tb = np.fromfile(f2, dtype=rectype)
            dt = round(self.Tp/v_tb.shape[0],4)
            print("dt_calc", dt)
#             print("ex {}, dt{}".format(ex, dt))
#             self.x = np.arange(0, self.T, dt)/1000
        except:
            print("dt_calc_Error")
            traceback.print_exc()
        rho_m_li = np.empty(0)
        for n in range(N):
            for s in range(N):
                cpre_file = self.drc_syn+"cpre"+str(s)+"-"+str(n)+"_"+str(0)+".bin"
                if os.path.exists(cpre_file):
                    rho = self.rho_of_exp(s, n)
                    rho_m = np.mean(rho[int(start/dt):])
                    rho_m_li = np.append(rho_m_li, rho_m)
        return np.mean(rho_m_li)
    
    def ave_rho2(self, N,  start):
        try:
            v_file = self.drc_neu +"N"+str(0)+"_"+str(0)+".bin" 
            f2= open(v_file, "rb")
            rectype = np.dtype(np.float64)
            v_tb = np.fromfile(f2, dtype=rectype)
            dt = round(self.Tp/v_tb.shape[0],4)
            print("dt_calc", dt)
#             print("ex {}, dt{}".format(ex, dt))
#             self.x = np.arange(0, self.T, dt)/1000
        except:
            print("dt_calc_Error")
            traceback.print_exc()
        syN=0
        for n in range(N):
            for s in range(N):
                cpre_file = self.drc_syn+"cpre"+str(s)+"-"+str(n)+"_"+str(0)+".bin"
                if os.path.exists(cpre_file):
                    syN+=1
                    
        rho_m_li = np.zeros((syN, int((self.T-start)/dt)))
        c=0
        for n in range(N):
            for s in range(N):
                cpre_file = self.drc_syn+"cpre"+str(s)+"-"+str(n)+"_"+str(0)+".bin"
                if os.path.exists(cpre_file):
                    rho = self.rho_of_exp(s, n)
                    rho_m_li[c]=rho[int(start/dt):]
                    c+=1
#                     rho_m = np.mean(rho[int(start/dt):])
#                     rho_m_li = np.append(rho_m_li, rho_m)
        return rho_m_li
    
    def raster(self, N, ex, start_T, T_w):
        
        start_num = int(start_T/Tp)
        tbs = int((T_w-start_T)/self.Tp)
   
        spts = np.zeros((N, int((T_w-start_T)/self.dt)), dtype = "int8")
        for n in range(N):
            try:
                for tb in range(tbs):
                    v_file = self.drc_neu+ "ex"+str(ex)+"_"+"ex_in"+str(ex_in)+"_"+"N"+str(n)+"_"+str(start_num+tb)+".bin" 
                    f2= open(v_file, "rb")
                    rectype = np.dtype(np.float64)
                    v_tb = np.fromfile(f2, dtype=rectype)
                    if tb==0:
                        v=v_tb
                    else:
                        v=np.append(v, v_tb)
                    f2.close()
                    
                pattern, maxfre, sp_c, peak_np = self.fre_spike(v[int(start_T/self.dt):int(T_w/self.dt-start_T/self.dt)])
           
                if maxfre!=0 and sp_c!=0:
                    if pattern!= "ERROR" and pattern != "EXCLUDED": 
                    # if pattern == "SWS" or pattern == "SWS_HIGH_FR" or pattern == "AWAKE" or pattern == "AWAKE_HIGH_FR" or pattern == "SWS_FEW_SPIKES" or pattern == "RES":
                        spts[n]= peak_np
            except:
                print("make_sp_np_error")
                traceback.print_exc()
        
        plt.figure(figsize=(23, 10))
#         Lt = int((T_w-start_T)/self.dt)
        x2= np.arange(0, (T_w-start_T), self.dt) #np.arange(0,Lt,1)
        
        #color_set = ['r', 'b', 'k', 'orange', 'c']
        for i in range(spts.shape[0]):
            t_sp = x2[spts[i, :] > 0.5]   # spike times
            plt.plot(t_sp/1000, i*np.ones(len(t_sp)), '.',
                 ms=8, markeredgewidth=0.1, color="black")
        plt.xlabel('Time (sec)',fontsize = fs)
        plt.ylabel('Neuron ID', fontsize = fs)
        #plt.xlim(0, 1)
        plt.xticks(fontsize = fs_l)
        plt.yticks(fontsize = fs_l)
        plt.show()
        
#     def histeri(self, rho_inter, cv_T, cv_w, moving, N_num, pre_syn, ex):
#         rho_count = int((T-cv_T)/rho_inter +1)
#         his = np.zeros((rho_count, 2))
#         init = int(cv_T/2/self.dt)
        
#         rho_array = self.rho_of_exp(pre_syn, N_num, ex)
# #         rho_ini = rho_array[init]
#         print(rho_array)
# #         v = self.v_of_exp(n, ex)
#         for i in range(rho_count):
#             it = int(init+ i*rho_inter/self.dt)
#             rho = rho_array[it]
#             sp_np = np.zeros((self.NE, int(cv_T/self.dt)), dtype = "int8")
#             for n in range(self.NE):
#                 v = self.v_of_exp(n, ex)
#                 v_range = v[int(it-(cv_T/2/self.dt)):int(it+(cv_T/2/self.dt))]
#                 pattern, maxfre, sp_c, peak_np = self.fre_spike(v_range, cv_T, self.dt)
#                 sp_np[n] = peak_np
#             cv = self.calc_cv(self.NE, sp_np, cv_T, cv_w,  moving)
#             his[i][0] = rho
#             his[i][1] = cv
#         df_his=pd.DataFrame(his, columns=["rho", "cv"])
#         print(df_his)
#         df_his.to_excel(self.drc_neu + "histerisis_{}_{}.xlsx".format(pre_syn, N_num))
#         return df_his
    def calc_spts_100ms(self, N, ex,v_off):
        T_w=100  #ms
        tc = int(self.T/T_w)
        print("step_num", tc)
        off_c=int(v_off/self.Tp)
        w_c=int(T_w/self.Tp)
        print("w_c", w_c)
#         ex =round(ex,2)
        ex_in = (ex + self.ex_diff)
        sp_sums = np.zeros(tc)
        
        flag=0
        for it in range(tc):
            spts = np.zeros((N, int((T_w)/self.dt)), dtype = "int8")
            for n in range(N):
#                 print("n", n)
#                 try:
                for tb in range(off_c+it*w_c, off_c+(it+1)*w_c):
#                     print("tb", tb)
                    v_file = self.drc_neu+ "ex"+str(ex)+"_"+"ex_in"+str(ex_in)+"_"+"N"+str(n)+"_"+str(int(tb))+".bin" 
                    f2= open(v_file, "rb")
                    rectype = np.dtype(np.float64)
                    v_tb = np.fromfile(f2, dtype=rectype)
                    if tb==off_c+it*w_c:
                        v=v_tb
                    else:
                        v=np.append(v, v_tb)
                    f2.close()
                pattern, maxfre, sp_c, peak_np = self.fre_spike(v)
                
                try:
                    if maxfre!=0 and sp_c!=0:
                        if pattern != "ERROR" and pattern != "EXCLUDED": 
                            spts[n]= peak_np
                except:
                    print("file_Error, cant make DF")
                    traceback.print_exc()
                    flag = 1
#             print(spts.shape)
            total_sps = np.sum(spts)
            sp_sums[it] = total_sps
            
        return sp_sums
    
    def mean_fre(self, N, start):
#         try:
#             v_file = self.drc_neu +"N"+str(n)+"_"+str(tb)+".bin" 
#             f2= open(v_file, "rb")
#             rectype = np.dtype(np.float64)
#             v_tb = np.fromfile(f2, dtype=rectype)
#             dt = round(self.Tp/v_tb.shape[0],4)
#             print("dt_calc", dt)
#             print("ex {}, dt{}".format(ex, dt))
#             self.x = np.arange(0, self.T, dt)/1000
#         except:
#             print("file_Error, ex{}".format(ex))
#             traceback.print_exc()
        spN=np.zeros(N, dtype="int64")
        flag = 0
        for n in range(N):
            for tb in range(self.tbs):
                v_file = self.drc_neu +"N"+str(n)+"_"+str(tb)+".bin" 
                f2= open(v_file, "rb")
                rectype = np.dtype(np.float64)
                v_tb = np.fromfile(f2, dtype=rectype)
                if tb==0:
                    v=v_tb
                else:
                    v=np.append(v, v_tb)
                f2.close()
            pattern, maxfre, sp_c, peak_np = self.fre_spike(v, start)
            spN[n]=sp_c/((self.T-start)/1000)
            if sp_c==0:
                flag=1
                break
        m_fre= np.mean(spN)
#         print("mean spike fre", m_fre)
        return m_fre, flag
            
            
    
   
    def calc_cvs(self, ex, N, T_w, cv_w, moving, moving_cv, v_off):
#         tc =  int((self.T-T_w)/moving+1)
        tc =  int((self.tbs*self.Tp - T_w)/moving+1)
        
        cv_li = np.zeros(tc)
        off_c=int(v_off/self.Tp)
#         ex =round(ex,2)
#         ex_in = round((ex + self.ex_diff), 2)
            
        spts = np.zeros((N, int(self.tbs*self.Tp/self.dt)), dtype = "int8")
        for n in range(N):
            for tb in range(self.tbs):
#                     print("tb", tb)
                v_file = self.drc_neu+ "ex"+str(ex)+"_"+"ex_in"+str(ex_in)+"_"+"N"+str(n)+"_"+str(int(tb))+".bin" 
                f2= open(v_file, "rb")
                rectype = np.dtype(np.float64)
                v_tb = np.fromfile(f2, dtype=rectype)
                if tb==0:
                    v=v_tb
                else:
                    v=np.append(v, v_tb)
                f2.close()
#                         if n<self.NE:
#                             if n==0:
#                                 vm_li = v
#                             else:
#                                 vm_li=np.append(vm_li, v)

            pattern, maxfre, sp_c, peak_np = self.fre_spike(v)
            spts[n]= peak_np
        offset = 0
        for i in range(tc):
            cv = self.calc_cv(N, spts, T_w, offset, cv_w,  moving_cv)
            print("{}, CV:{}/".format(i, cv))
            cv_li[i]=cv
            offset += int(moving/self.dt)
            
        return cv_li
    
    def calc_cv(self, N, spts, T_w, offset, cv_w,  moving):
        wc = int((T_w-cv_w)/moving+1)
#         print("wc", wc)
        sp_li = np.zeros(wc)
        for m in range(wc):
            if m*moving/self.dt>=0 and m*moving/self.dt + cv_w/self.dt  <= (T_w)/self.dt:
                sp_sum = np.sum(spts[0:N,int(m*moving/self.dt)+offset:int(m*moving/self.dt + cv_w/self.dt)+offset])
                sp_li[m]=sp_sum
#         print("sp_li", sp_li)
        if np.mean(sp_li)!=0:
            sps_cv = np.var(sp_li)/np.mean(sp_li)
        else:
            sps_cv =0
        return sps_cv
        


    def calc_values_lr(self, ex, T_w, cv_w, moving, v_off, calc =True, graph = True):
        tc = int(self.T/T_w)
        
        
        off_c=int(v_off/self.Tp)
        w_c=int(T_w/self.Tp)
#         print("w_c", w_c)
#         ex =round(ex,2)
        ex_in = (ex + self.ex_diff)
        
        flag=0
        for it in range(tc):
            print("it", it)
#             pattern_li = []
#             pattern_id=[]
            wake_c = 0
            sleep_c = 0
            wake_ex_c =0
            sleep_ex_c=0

            sp_fre_li = np.zeros(self.N)
            
            spts = np.zeros((self.N, int((T_w)/self.dt)), dtype = "int8")
            for n in range(self.N):
#                 print("n", n)
#                 try:
                for tb in range(off_c+it*w_c, off_c+(it+1)*w_c):
#                     print("tb", tb)
                    v_file = self.drc_neu+ "ex"+str(ex)+"_"+"ex_in"+str(ex_in)+"_"+"N"+str(n)+"_"+str(int(tb))+".bin" 
                    f2= open(v_file, "rb")
                    rectype = np.dtype(np.float64)
                    v_tb = np.fromfile(f2, dtype=rectype)
                    if tb==0:
                        v=v_tb
                    else:
                        v=np.append(v, v_tb)
                    f2.close()
#                         if n<self.NE:
#                             if n==0:
#                                 vm_li = v
#                             else:
#                                 vm_li=np.append(vm_li, v)



                pattern, maxfre, sp_c, peak_np = self.fre_spike(v[0:int(T_w/self.dt)])
                

                if graph==True:
                    if self.N > 60:
                        if n%30 ==0:
                            plt.figure(figsize=(15, 4))
                            plt.title("Neuron{}".format(n))
                            plt.plot(v)   #time[0:int(T/dt)-1],
                            plt.ylim(-100, 40)
                            plt.xlabel('Time (sec)',fontsize = fs)
                    #             plt.xlim(0,2)
                                #plt.ylim(0,1)
                            plt.xticks(fontsize = fs)
                            plt.yticks(fontsize = fs)

                            plt.show()

                    else:
                        plt.figure(figsize=(15, 4))
                        plt.title("Neuron{}".format(n))
                        plt.plot(v)   #time[0:int(T/dt)-1],
                        plt.ylim(-100, 40)
                        plt.xlabel('Time (sec)',fontsize = fs)
                #             plt.xlim(0,2)
                            #plt.ylim(0,1)
                        plt.xticks(fontsize = fs)
                        plt.yticks(fontsize = fs)

                        plt.show()


                if np.any(np.isinf(v)) or np.any(np.isnan(v)):
                    flag = 1
                    print("nan_Error, cant make DF")
                    self.df_np[it][0]=1
                    self.df_np[it][1:] = -1
                    break  #N
                    # continue
        #                                                     
                if flag==1:
                    continue

                if pattern.name == "SWS" or pattern.name == "SWS_HIGH_FR":
                    sleep_c+=1
                    if n<self.NE:
                        sleep_ex_c+=1
                if pattern.name == "AWAKE" or pattern.name == "AWAKE_HIGH_FR":
                    wake_c+=1
                    if n<self.NE:
                        wake_ex_c+=1

#                     pattern_li.append(pattern.name)
                sp_fre_li[n]=(sp_c/(self.T-v_off)*1000)
                # print(pattern.name)
                # print("spike_count: ", sp_c)
                # print("maxfre: ", maxfre)

                try:
                    if maxfre!=0 and sp_c!=0:
                        if pattern.name != "ERROR" and pattern.name != "EXCLUDED": 
                        # if pattern.name == "SWS" or pattern.name == "SWS_HIGH_FR" or pattern.name == "AWAKE" or pattern.name == "AWAKE_HIGH_FR" or pattern.name == "SWS_FEW_SPIKES" or pattern.name == "RES":
                            spts[n]= peak_np
        #                                                             print("{}, sp_add".format(n))
#                                     pattern_id.append(n)
                except:
                    print("file_Error, cant make DF")
                    traceback.print_exc()
                    flag = 1
                    self.df_np[it][0]=2
                    self.df_np[it][1:] = -1
                    break  #N end


            if flag!=1 and calc ==True:
                m_sp_fre =np.mean(sp_fre_li)
                var_sp_fre =np.var(sp_fre_li)
                m_sp_fre_ex =np.mean(sp_fre_li[0:self.NE])
                var_sp_fre_ex =np.var(sp_fre_li[0:self.NE])
                sleep_per = sleep_c/(self.N)*100
                wake_per = wake_c/(self.N)*100
                sleep_per_ex = sleep_ex_c/(self.NE)*100
                wake_per_ex = wake_ex_c/(self.NE)*100


                self.df_np[it][0] = 0
                self.df_np[it][1] = wake_per
                self.df_np[it][2] = sleep_per
                self.df_np[it][3] = m_sp_fre
                self.df_np[it][4] = var_sp_fre
                self.df_np[it][6] = wake_per_ex
                self.df_np[it][7] = sleep_per_ex
                self.df_np[it][8] = m_sp_fre_ex
                self.df_np[it][9] = var_sp_fre_ex
                print("mean_fre_ex", m_sp_fre_ex)
                print("var_sp_fre_ex", var_sp_fre_ex)
                print("%sleep_ex", sleep_per_ex)
                print("%wake_ex", wake_per_ex)

                sps_cv = self.calc_cv(self.N, spts, T_w, cv_w,  moving)
    #             print("CV_sp_count/w", sps_cv)
                self.df_np[it][5] = sps_cv
                sps_cv_ex = self.calc_cv(self.NE, spts, T_w, cv_w,  moving)
                print("CV_sp_count_ex/w", sps_cv_ex)
                self.df_np[it][10] = sps_cv_ex
        if flag!=1 and calc ==True:   
            df_cc = pd.DataFrame(self.df_np, columns=["%nan","%sleep","%wake","mean_spike_fre","var_spike_fre",  "cv", "%sleep_ex","%wake_ex","mean_spike_fre_ex","var_spike_fre_ex", "cv_ex"])
            df_cc.to_excel(self.drc_neu + "cc_exp{}.xlsx".format(ex))
            print(df_cc)
        return df_cc
            
    
    def fre_spike(self, v, start):
#         v=v[0:int(T/dt)]
        try:
            dt = round(self.T/v.shape[0],4)
#             print("dt_calc", dt)
#             print("ex {}, dt{}".format(ex, dt))
#             self.x = np.arange(0, self.T, dt)/1000
        except:
            print("dt_calc_Error")
            traceback.print_exc()

        T = int(v.shape[0]*dt-start) #ms
        if np.any(np.isinf(v)) or np.any(np.isnan(v)):
            print("fre_spike: nan")
            return "Exclude", 0, 0, np.zeros(int(T/dt), dtype="int8")
        else:
            wavech = anmodel.analysis.WaveCheck(1000)
            pattern = wavech.pattern(v, T, dt, pr=False)
            detv: np.ndarray = signal.detrend(v)
            max_potential: float = max(detv)
            f, spw = periodogram(detv, fs=1/dt*1000)
            maxamp: float = max(spw)
            nummax: int = spw.tolist().index(maxamp)
            maxfre: float = f[nummax]

    #         ntraverse: int = 0
    #         ms = 1
    #         for k in range(len(v)-1):
    #             if k+ms < len(v)-1:
    #                 if (v[k]+20) * (v[k+ms]+20) < 0:
    #                     ntraverse += 1

    #         nspike= (ntraverse/2)/(T/(2*1000))/20
            peaks = find_peaks(v[int(start/dt):], height=-20)[0]
#             print(v.shape)
            peak_np = np.zeros(int(T/dt), dtype="int8")
            peak_np[peaks]=1
#             print(peak_np.shape)
            return pattern.name, maxfre, len(peaks), peak_np
            
            
    




In [6]:
#make total files 
rdir = "../AN_network"

xl_ori =rdir+"/net_lr_data.xlsx"
df_ori = pd.read_excel(xl_ori, engine="openpyxl")
dates_ori = df_ori["date_i"]

model_pre = "AN"
op="_net_effi"


bifur = "_pre"
lr = "Anti-STDP"
lr_p = "a0.7t50th0.6"
NE = 64
Ip = 20
logs=[0.1, 0.5, 1.0]

con_ex_ex = 0.01
con_ex_in = 0.01
con_in_ex = 0.01
con_in_in = 0.01
NI = int(NE*Ip/(100-Ip))
N=NE+NI
con_M = 1.0
con_M_in = 1.0
rho_ini = 0.5
T =  60000#ms
Tp= 10000
dt = 0.05
smin_ampa=0.5
smax_ampa=1.5
init = "r"
seed = 4
con_log = True


model_name = model_pre + op + bifur +"_"+lr +"_w"
model_name_s = model_pre + op + bifur +"_"+lr +"_s"

sdir =rdir+ "/net_lr/"

df_num=[]

for log in logs:
    con_ex_ex = log
    con_ex_in =log
    con_in_ex = log
    con_in_in = log
    
    T_dir = sdir + "{}{}{}_N{}_{}_{}_conlog{}/".format(model_pre, op, bifur, NE+NI, lr, lr_p, log)
#     print(T_dir)
    try:
        os.makedirs(T_dir)
    except FileExistsError:
        print("{} is already exist".format(T_dir)) 

    savef_T = T_dir +"wake.csv" 
    savef_Ts = T_dir +"sleep.csv"
    savef_T_cv = T_dir +"wake_cv.csv" 
    savef_T_cv_s = T_dir +"sleep_cv.csv"

    
    drc =rdir+"/con_cu_i/{}/NE{}_NI{}/conM{}_conSD{}_conMin{}_conSDin{}/sminampa{}_smaxampa{}/{}_{}/".format(model_name,NE, NI,con_M, con_ex_ex,con_M_in, con_in_in,smin_ampa,smax_ampa,lr,lr_p)
    drc_s =rdir+"/con_cu_i/{}/NE{}_NI{}/conM{}_conSD{}_conMin{}_conSDin{}/sminampa{}_smaxampa{}/{}_{}/".format(model_name_s,NE, NI,con_M, con_ex_ex,con_M_in, con_in_in,smin_ampa,smax_ampa,lr,lr_p)


    date_is = os.listdir(drc)
    print(date_is)
    
    fre_w_li = []
    fre_s_li = []
    effi_w_li=[]
    effi_s_li=[]
    cv_w_li=[]
    cv_s_li=[]
    diff_li = []
    dates_ana=[]
    
#     for n, date_i in enumerate(date_is):
    c=0
    for n, date_i in enumerate(dates_ori):

#         if bifur =="_cav":
#             if date_i == "2023_3_29_0_0" or date_i=="2023_4_27_0_4" or date_i=="2023_6_30_4_2" or date_i=="2023_4_16_0_6":
#                 continue
    #

        savedir =  drc + date_i + "/"
        savedir_s =  drc_s + date_i +"/"
        savef=savedir + "wake.csv"
        savef_s=savedir_s + "sleep.csv"
        savef_cv=savedir + "wake_cv.csv"
        savef_cv_s=savedir_s + "sleep_cv.csv"

        fref=savedir + "m_wake_fre.csv"
        fref_s=savedir_s + "m_sleep_fre.csv"


        if not os.path.exists(fref) or not os.path.exists(fref_s):
            continue
        
        fre_w = np.genfromtxt(fref, delimiter=',', dtype=np.float64)
        fre_s = np.genfromtxt(fref_s, delimiter=',', dtype=np.float64)
      
        

        if fre_w==-1 or fre_s==-1 or np.abs(fre_w-fre_s)>2.0:
            continue
            
        if not os.path.exists(savef) or not os.path.exists(savef_s) or not os.path.exists(savef_cv) or not os.path.exists(savef_cv_s):
            continue
            
        
        effis = np.genfromtxt(savef, delimiter=',', dtype=np.float64)

        if not os.path.exists(savef_T):
            np.savetxt(savef_T, effis, delimiter=',')
        elif c==0 and os.path.exists(savef_T):
            np.savetxt(savef_T, effis, delimiter=',')  #create new file 

        elif c!=0 and os.path.exists(savef_T):
            pre =  np.genfromtxt(savef_T, delimiter=',', dtype=np.float64)
            total =np.vstack([pre, effis])
#                 print(total)
            np.savetxt(savef_T, total, delimiter=',')

        #CV
        effis_cv = np.genfromtxt(savef_cv, delimiter=',', dtype=np.float64)

        if not os.path.exists(savef_T_cv):
            np.savetxt(savef_T_cv, effis_cv, delimiter=',')
        elif c==0 and os.path.exists(savef_T_cv):
            np.savetxt(savef_T_cv, effis_cv, delimiter=',')  #create new file 

        elif c!=0 and os.path.exists(savef_T_cv):
            pre_cv =  np.genfromtxt(savef_T_cv, delimiter=',', dtype=np.float64)
            total_cv =np.vstack([pre_cv, effis_cv])
#                 print(total_cv)
            np.savetxt(savef_T_cv, total_cv, delimiter=',')


        effis_s = np.genfromtxt(savef_s, delimiter=',', dtype=np.float64)

        if not os.path.exists(savef_Ts):
            np.savetxt(savef_Ts, effis_s, delimiter=',')

        elif c==0 and os.path.exists(savef_Ts):
            np.savetxt(savef_Ts, effis_s, delimiter=',')  #create

        elif c!=0 and os.path.exists(savef_Ts):
            pre_s =  np.genfromtxt(savef_Ts, delimiter=',', dtype=np.float64)

            total_s =np.vstack([pre_s, effis_s])
#                 print(total_s)
            np.savetxt(savef_Ts, total_s, delimiter=',')

        #CV
        effis_cv_s = np.genfromtxt(savef_cv_s, delimiter=',', dtype=np.float64)

        if not os.path.exists(savef_T_cv_s):
            np.savetxt(savef_T_cv_s, effis_cv_s, delimiter=',')
        elif c==0 and os.path.exists(savef_T_cv_s):
            np.savetxt(savef_T_cv_s, effis_cv_s, delimiter=',')  #create new file 

        elif c!=0 and os.path.exists(savef_T_cv_s):
            pre_cv_s =  np.genfromtxt(savef_T_cv_s, delimiter=',', dtype=np.float64)
            total_cv_s =np.vstack([pre_cv_s, effis_cv_s])
#                 print(total_cv_s)
            np.savetxt(savef_T_cv_s, total_cv_s, delimiter=',')

        fre_w_li.append(fre_w)
        fre_s_li.append(fre_s)
        diff_li.append(fre_s-fre_w)
        effi_w_li.append(effis[1])
        effi_s_li.append(effis_s[1])
        cv_w_li.append(effis_cv[1])
        cv_s_li.append(effis_cv_s[1])
        dates_ana.append(date_i)
        c+=1
        
    
    df = pd.DataFrame(list(zip(dates_ana, fre_w_li,fre_s_li, effi_w_li, effi_s_li, cv_w_li, cv_s_li)), columns =["model ID","FR in wake states","FR in sleep states","mean synaptic efficacy in wake states","mean synaptic efficacy in sleep states","CV of synaptic efficacy in wake states","CV of synaptic efficacy in sleep states"])
    print(len(df))
    
    df_num.append(len(df))
#     print(df)
    df.to_excel("network_model_{}{}_SD{}_pnas_0130.xlsx".format(lr, bifur, log))


    mean_fre_w = np.mean(fre_w_li)
    mean_fre_s = np.mean(fre_s_li)
    mean_effi_w =  np.mean(effi_w_li)
    mean_effi_s =  np.mean(effi_s_li)
    mean_cv_w =  np.mean(cv_w_li)
    mean_cv_s =  np.mean(cv_s_li)
    mean_diff = np.mean(diff_li)

    std_fre_w = np.std(fre_w_li)
    std_fre_s = np.std(fre_s_li)
    std_effi_w =  np.std(effi_w_li)
    std_effi_s =  np.std(effi_s_li)
    std_cv_w =  np.std(cv_w_li)
    std_cv_s =  np.std(cv_s_li)
    std_diff = np.std(diff_li)

    mean_fre_sd_w = "{} ± {}".format(round(mean_fre_w, 3), round(std_fre_w, 3))
    mean_fre_sd_s = "{} ± {}".format(round(mean_fre_s, 3), round(std_fre_s, 3))                               
    mean_diff_sd = "{} ± {}".format(round(mean_diff, 3), round(std_diff, 3))
    mean_effi_sd_w = "{} ± {}".format(round(mean_effi_w, 3), round(std_effi_w, 3))
    mean_effi_sd_s = "{} ± {}".format(round(mean_effi_s, 3), round(std_effi_s, 3))
    mean_cv_sd_w = "{} ± {}".format(round(mean_cv_w, 3), round(std_cv_w, 3))
    mean_cv_sd_s = "{} ± {}".format(round(mean_cv_s, 3), round(std_cv_s, 3))

    mean_sd_li = [len(df), mean_fre_sd_w ,mean_fre_sd_s,mean_diff_sd,mean_effi_sd_w,mean_effi_sd_s,mean_cv_sd_w,mean_cv_sd_s]
    df_net_all = pd.DataFrame(mean_sd_li).T

    df_net_all.columns = ["number", "mean FR in wake states", "mean FR in sleep states", "mean difference in FR between wake and sleep states (sleep-wake)","mean synaptic efficacy in wake states","mean synaptic efficacy in sleep states" , "CV of synaptic efficacy in wake states","CV of synaptic efficacy in sleep states"] #,"mean_cv_w","std_cv_w", "mean_cv_s", "std_cv_s"]
    print(df_net_all)
    df_net_all.to_excel("network_model_{}{}_SD{}_total_pnas_0130.xlsx".format(lr, bifur, log))
    
print(df_num)

['2023_9_29_10_13', '2023_9_29_10_17', '2023_10_2_2_7', '2023_10_1_2_5', '2023_10_4_7_1', '2023_10_6_7_0', '2023_10_6_7_1', '2023_6_4_2_6', '2023_6_4_2_7', '2023_6_4_2_10', '2023_7_9_2_5', '2023_7_21_0_0', '2023_9_30_7_11', '2023_8_8_0_6', '2023_10_6_7_19', '2023_10_4_7_12', '2023_4_21_0_7', '2023_4_13_0_4', '2023_4_13_0_10', '2023_8_26_8_3', '2023_10_13_7_16', '2023_10_17_7_2', '2023_10_17_7_21', '2023_9_25_7_22', '2023_10_30_7_4', '2023_11_4_7_12', '2023_11_4_8_14', '2023_10_20_7_21', '2023_10_23_7_13', '2023_9_25_7_28', '2023_9_23_10_6', '2023_11_4_8_16', '2023_11_5_7_7', '2023_11_5_8_2', '2023_11_5_8_3', '2023_11_5_8_9', '2023_9_27_10_4', '2023_9_27_10_20', '2023_9_27_10_24', '2023_9_30_7_0', '2023_11_5_8_15', '2023_11_5_8_16', '2023_11_10_8_69', '2023_11_5_8_18', '2023_11_3_2_32', '2023_10_30_8_16', '2023_11_7_8_6', '2023_10_30_8_19', '2023_10_30_8_30', '2023_11_10_8_22', '2023_10_27_7_3', '2023_11_10_8_88', '2023_11_15_7_4', '2023_11_15_8_0', '2023_11_15_8_18', '2023_11_15_8_30',

In [ ]:
# comparing sleep and wake by learning rules

bifurs =["nmdar", "cav", "pre"]

model_pre = "AN"
op="_net_effi"

lr = "Anti-STDP"
lr_p = "a0.7t50th0.6"
NE = 64
Ip = 20
NI = int(NE*Ip/(100-Ip))
N=NE+NI

logs=[0.01, 0.1, 0.5, 1.0]
con_log=True

lr_li =[lr] 
sw = ["sleep", "wake"]

for bifur in bifurs:
    print(bifur)

    model_name = model_pre + op +"_"+ bifur +"_"+lr +"_w"
    model_name_s = model_pre + op + "_"+bifur +"_"+lr +"_s"
    sdir = rdir+"/net_lr/"


    box_pairs = []
    for h, log in enumerate(logs):
        pair_sw = []
        for i, sw_i in enumerate(sw):
            pair_sw.append((log, sw_i))
        box_pairs.append((pair_sw[0], pair_sw[1]))


    fs = 18
    fs_l = 15
    Hz="4.0"
    colors =[sns.xkcd_rgb["magenta"]]

    dfs2=[]

    n_li=[]

    for log in logs:
        con_ex_ex = log
        con_ex_in =log
        con_in_ex = log
        con_in_in = log

        if log==0.01:
            T_dir = sdir + "{}{}_{}_N{}_{}_{}/".format(model_pre, op, bifur, NE+NI, lr, lr_p)
        else:
            T_dir = sdir + "{}{}_{}_N{}_{}_{}_conlog{}/".format(model_pre, op, bifur, NE+NI, lr, lr_p, log)

        for m, lr in enumerate(lr_li):
            dsb_w = T_dir+"/wake.csv"
            dsb_s = T_dir+"/sleep.csv"
            synv_w= np.genfromtxt(dsb_w, delimiter=",", dtype=np.float64)
            print("wake", lr,  len(synv_w), np.mean(synv_w,axis = 0))
            synv_s= np.genfromtxt(dsb_s, delimiter=",", dtype=np.float64)
            print("sleep", lr,  len(synv_s), np.mean(synv_s,axis = 0))

            df_w = pd.DataFrame(synv_w, columns=["order","mean_effi"])
            df_s = pd.DataFrame(synv_s, columns=["order","mean_effi"])
            df_w["lr"] = lr
            df_s["lr"] = lr
            df_w["log"] = log
            df_s["log"] = log
            df_w["state"] = "wake"
            df_s["state"] = "sleep"

            df_s["Hz"] = Hz
            df_w["Hz"] = Hz

            #print(df)

        #             if l==0 and n==0:
        #                 dfs = df
        #             else:
            n_li.append(len(df_s))
            dfs = pd.concat([df_s, df_w])
#             print(dfs)
            dfs2.append(dfs)


    dfs_all=pd.concat(dfs2, axis=0)

    plt.figure(figsize=(10,10))

    x = "log"
    y = "mean_effi"
    hue = "state"
    hue_order = sw
    ax = sns.boxplot(data=dfs_all,palette=[colors[m], sns.xkcd_rgb["white"]], x=x, y=y, hue=hue)
    add_stat_annotation(ax, data=dfs_all, x=x, y=y, hue=hue,comparisons_correction=None,

                                         box_pairs = box_pairs,

                        test='t-test_ind', text_format='star', loc='outside', fontsize=fs, verbose=2)

    plt.yticks(fontsize = fs_l)
    plt.xticks(fontsize = fs)
    ax.set_yticks(np.arange(0,1.1,0.25))
    ax.set_ylim(0, 1.1)

    plt.ylabel("mean synaptic efficacy", fontsize = fs)
    plt.xlabel("SD of nember of synapses", fontsize = fs)


    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.grid(False)

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=fs_l)
    plt.tight_layout()

    plt.savefig("boxplot_{}_N{}_{}_{}_conlog_all.SVG".format(model_name,  N, lr, lr_p))
    plt.show()

    print("n", n_li)

In [ ]:
# comparing sleep and wake by learning rules  CV
#mean last point, mean syn efficacy
bifurs =["nmdar", "cav", "pre"]

model_pre = "AN"
op="_net_effi"

lr = "Anti-STDP"
lr_p = "a0.7t50th0.6"
NE = 64
Ip = 20
NI = int(NE*Ip/(100-Ip))
N=NE+NI

logs=[0.01, 0.1, 0.5, 1.0]
con_log=True

lr_li =[lr] 
sw = ["sleep", "wake"]

for bifur in bifurs:
    print(bifur)

    model_name = model_pre + op +"_"+ bifur +"_"+lr +"_w"
    model_name_s = model_pre + op + "_"+bifur +"_"+lr +"_s"
    sdir = rdir+"/net_lr/"


    box_pairs = []
    for h, log in enumerate(logs):
        pair_sw = []
        for i, sw_i in enumerate(sw):
            pair_sw.append((log, sw_i))
        box_pairs.append((pair_sw[0], pair_sw[1]))


    fs = 18
    fs_l = 15
    Hz="4.0"
    colors =[sns.xkcd_rgb["magenta"]]

    dfs2=[]

    n_li=[]

    for log in logs:
        con_ex_ex = log
        con_ex_in =log
        con_in_ex = log
        con_in_in = log

        if log==0.01:
            T_dir = sdir + "{}{}_{}_N{}_{}_{}/".format(model_pre, op, bifur, NE+NI, lr, lr_p)
        else:
            T_dir = sdir + "{}{}_{}_N{}_{}_{}_conlog{}/".format(model_pre, op, bifur, NE+NI, lr, lr_p, log)

        for m, lr in enumerate(lr_li):
            dsb_w = T_dir+"/wake_cv.csv"
            dsb_s = T_dir+"/sleep_cv.csv"
            synv_w= np.genfromtxt(dsb_w, delimiter=",", dtype=np.float64)
            print("wake", lr,  len(synv_w), np.mean(synv_w,axis = 0))
            synv_s= np.genfromtxt(dsb_s, delimiter=",", dtype=np.float64)
            print("sleep", lr,  len(synv_s), np.mean(synv_s,axis = 0))

            df_w = pd.DataFrame(synv_w, columns=["order","CV"])
            df_s = pd.DataFrame(synv_s, columns=["order","CV"])
            df_w["lr"] = lr
            df_s["lr"] = lr
            df_w["log"] = log
            df_s["log"] = log
            df_w["state"] = "wake"
            df_s["state"] = "sleep"

            df_s["Hz"] = Hz
            df_w["Hz"] = Hz

            #print(df)

        #             if l==0 and n==0:
        #                 dfs = df
        #             else:
            n_li.append(len(df_s))
            dfs = pd.concat([df_s, df_w])
#             print(dfs)
            dfs2.append(dfs)


    dfs_all=pd.concat(dfs2, axis=0)

    plt.figure(figsize=(10,10))

    x = "log"
    y = "CV"
    hue = "state"
    hue_order = sw
    ax = sns.boxplot(data=dfs_all,palette=[colors[m], sns.xkcd_rgb["white"]], x=x, y=y, hue=hue)
    add_stat_annotation(ax, data=dfs_all, x=x, y=y, hue=hue,comparisons_correction=None,

                                         box_pairs = box_pairs,

                        test='t-test_ind', text_format='star', loc='outside', fontsize=fs, verbose=2)

    plt.yticks(fontsize = fs_l)
    plt.xticks(fontsize = fs)
    ax.set_yticks(np.arange(0,1.1,0.25))
    ax.set_ylim(0, 1.1)

    plt.ylabel("mean synaptic efficacy", fontsize = fs)
    plt.xlabel("SD of nember of synapses", fontsize = fs)


    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.grid(False)

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0, fontsize=fs_l)
    plt.tight_layout()

    plt.savefig("boxplot_{}_N{}_{}_{}_conlog_all_CV.SVG".format(model_name,  N, lr, lr_p))
    plt.show()

    print("n", n_li)